In [3]:
import numpy as np
import joblib

from tensorflow.keras.models import load_model

In [7]:
from pathlib import Path

BASE_DIR = Path("..")

MODEL_PATH = BASE_DIR / "models" / "multi_disease_ann.h5"
ENCODER_PATH = BASE_DIR / "models" / "disease_label_encoder.pkl"
FEATURE_PATH = BASE_DIR / "models" / "disease_feature_names.pkl"

model = load_model(MODEL_PATH)

label_encoder = joblib.load(
    ENCODER_PATH
)

feature_names = joblib.load(
    FEATURE_PATH
)

print("Model loaded successfully.")
print("Number of features:", len(feature_names))
print("Number of diseases:", len(label_encoder.classes_))
print("Model input shape:", model.input_shape)

Model loaded successfully.
Number of features: 328
Number of diseases: 659
Model input shape: (None, 328)


In [8]:
def predict_disease(symptoms, top_k=5):

    # Create empty input vector
    input_data = np.zeros(
        len(feature_names),
        dtype=np.float32
    )

    # Set selected symptoms to 1
    for symptom in symptoms:

        if symptom in feature_names:

            index = feature_names.index(symptom)

            input_data[index] = 1

        else:

            print(
                f"Warning: '{symptom}' not found in dataset."
            )

    # Reshape for ANN
    input_data = input_data.reshape(1, -1)

    # Predict probabilities
    probabilities = model.predict(
        input_data,
        verbose=0
    )[0]

    # Get top K predictions
    top_indices = np.argsort(
        probabilities
    )[-top_k:][::-1]

    results = []

    for index in top_indices:

        disease = label_encoder.inverse_transform(
            [index]
        )[0]

        probability = probabilities[index]

        results.append({
            "disease": disease,
            "probability": float(probability)
        })

    return results

In [9]:
print(feature_names[:50])

['anxiety_and_nervousness', 'depression', 'shortness_of_breath', 'depressive_or_psychotic_symptoms', 'sharp_chest_pain', 'dizziness', 'insomnia', 'abnormal_involuntary_movements', 'chest_tightness', 'palpitations', 'irregular_heartbeat', 'breathing_fast', 'hoarse_voice', 'sore_throat', 'difficulty_speaking', 'cough', 'nasal_congestion', 'throat_swelling', 'diminished_hearing', 'lump_in_throat', 'throat_feels_tight', 'difficulty_in_swallowing', 'skin_swelling', 'retention_of_urine', 'groin_mass', 'leg_pain', 'hip_pain', 'suprapubic_pain', 'blood_in_stool', 'lack_of_growth', 'emotional_symptoms', 'elbow_weakness', 'back_weakness', 'symptoms_of_the_scrotum_and_testes', 'swelling_of_scrotum', 'pain_in_testicles', 'flatulence', 'pus_draining_from_ear', 'jaundice', 'mass_in_scrotum', 'white_discharge_from_eye', 'irritable_infant', 'abusing_alcohol', 'fainting', 'hostile_behavior', 'drug_abuse', 'sharp_abdominal_pain', 'feeling_ill', 'vomiting', 'headache']


In [10]:
symptoms = [
    "fever",
    "cough",
    "headache"
]

In [11]:
results = predict_disease(
    symptoms,
    top_k=5
)

for i, result in enumerate(results, start=1):

    print(
        f"{i}. {result['disease']} "
        f"-> {result['probability'] * 100:.2f}%"
    )

1. interstitial lung disease -> 20.89%
2. abscess of the pharynx -> 11.04%
3. acute bronchitis -> 8.81%
4. common cold -> 8.49%
5. flu -> 7.67%


In [12]:
import random

random_symptoms = random.sample(
    feature_names,
    5
)

print("Selected symptoms:")
for symptom in random_symptoms:
    print("-", symptom)

results = predict_disease(
    random_symptoms,
    top_k=5
)

print("\nPredictions:")

for i, result in enumerate(results, start=1):

    print(
        f"{i}. {result['disease']} "
        f"-> {result['probability'] * 100:.2f}%"
    )

Selected symptoms:
- cramps_and_spasms
- eye_moves_abnormally
- pulling_at_ears
- nausea
- involuntary_urination

Predictions:
1. brachial neuritis -> 91.26%
2. yeast infection -> 8.22%
3. Rare Disease -> 0.40%
4. cystitis -> 0.07%
5. spina bifida -> 0.01%
